In [0]:
%sql 
USE flights_silver;

### Implementacja SCD2

Używamy logiki SCD2, aby zachować historię zmian atrybutów. Celowo powielamy wiersze, aby umożliwić połączenie historycznych faktów z poprawnmi danymi (np. nazwą lotniska), które wtedy obowiązywały. 

Do każego rekordu dopisujemy 3 kolumny: *is_current*, *start_time*, *end_time*, któe będą wskazywać na to, czy informacje są dalej aktualne, a jeśli nie są, to od kiedy do kiedy aktualne były. Nie znamy historycznej daty wejścia konkretnych rekordów do użycia, więc dla pierwszego załadaowania używamy "dummy date": 1900-01-01 (uzupełnianie *start_time* aktualną datą nie ma sensu, jako że dane dotyczą lotów z roku 2015). 

Dzięki temu podejściu, jeśli któryś z portów lotniczych zmieni nazwę, rekord zostanie zamknięty (*is_current* zostanie ustawione na False, a *end_time* przestanie byc nullem, a datą, do której poprzednia nazwa funkcjonowała). Dla danej linii do tabeli zostanie dopisany nowy wiersz, zawierający aktualne informacje.

In [0]:
%python
from pyspark.sql.functions import lit, col, to_date

df_airports_raw = spark.table("flights_bronze.airports")

df_dim_airports = df_airports_raw.select(
    col('iata_code').alias("airport_key"),
    col('iata_code'),
    col('airport').alias("airport_name"),
    
    col('city'),
    col('state'),
    col('country'),
    col('latitude'),
    col('longitude'),

    lit(True).alias("is_current"),
    to_date(lit("1900-01-01")).alias("start_date"), # dummy data, bo nie wiemy od kiedy ważne
    to_date(lit(None)).alias("end_date") # wciąż jest aktywne => end_date = null
)

df_dim_airports.write.mode("overwrite").insertInto("flights_silver.dim_airports")

In [0]:
%python
display(df_dim_airports.limit(5))

airport_key,iata_code,airport_name,city,state,country,latitude,longitude,is_current,start_date,end_date
ABE,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.4404,true,1900-01-01,null
ABI,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.6819,true,1900-01-01,null
ABQ,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919,true,1900-01-01,null
ABR,ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183,true,1900-01-01,null
ABY,ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447,true,1900-01-01,null


### Implementacja procesu ładowania przyrostowego

Nowe dane łączymy jedynie z aktualnymi rekordami, nie biorąc pod uwagę tych historycznych. 

Jeśli zgadza się *iata_code* i *is_current* ustawione jest na True, możemy działać dalej. Jeśli któraś z wartości się zmieniła (nazwa, lokalizacja, pańswto, itp.), musimy zamknąć rekord -> *is_current* ustawić na False, a za *end_date* wstawić aktualną datę (datę, kiedy ładujemy nowe dane). 

Na koniec ładujemy nowe lub zmienione rekordy (*is_current*=True, *start_date*=dzisiaj, *end_date*=null)

In [0]:
%python
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit, current_date

airports_silver = "flights_silver.dim_airports" 
airports_bronze = spark.table("flights_bronze.airports") # aktualny stan danych w warstwie bronze

target_table = DeltaTable.forName(spark, airports_silver)

(target_table.alias("target")
 .merge(
   airports_bronze.alias("source"),
   # łaczymy tylko, jeśli zgadza się klucz biznesowy (iata code) i jeśli rekord jest aktywny
   "target.iata_code = source.iata_code AND target.is_current = true"
)
 # rekord istnieje, ale jakaś wartość się zmieniła -> zamykamy stary rekord
 .whenMatchedUpdate(
     condition=(
         (col("target.airport_name") != col("source.airport")) |
         (col("target.city") != col("source.city")) |
         (col("target.state") != col("source.state")) |
         (col("target.country") != col("source.country")) |
         (col("target.latitude") != col("source.latitude")) |
         (col("target.longitude") != col("source.longitude"))
     ),
     set={
         "is_current": lit(False), # już nieaktualne
         "end_date": current_date() # obecna data datą zamknięcia
     }
 )
 # rekord jest nowy lub został zamknięty -> tworzymy nowy
 .whenNotMatchedInsert(
     values={
         "airport_key": col("source.iata_code"),
         "iata_code": col("source.iata_code"),
         "airport_name": col("source.airport"),
         "city": col("source.city"),
         "state": col("source.state"),
         "country": col("source.country"),
         "latitude": col("source.latitude"),
         "longitude": col("source.longitude"),
         "is_current": lit(True),
         "start_date": current_date(),
         "end_date": lit(None).cast("date")
     }
 )
 .execute()
)


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]